# create sp for logging

In [2]:
use [sample];
GO

SELECT DB_NAME() AS db_name;
GO

Commands completed successfully.

(1 row affected)

db_name
-------
sample 
(1 row)

In [3]:
/*
IF OBJECT_ID('spLog', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spLog
    PRINT 'dbo.spLog DELETED'
END
IF OBJECT_ID('spInfo', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spInfo
    PRINT 'dbo.spInfo DELETED'
END
IF OBJECT_ID('spWarn', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spWarn
    PRINT 'dbo.spWarn DELETED'
END
IF OBJECT_ID('spError', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spError
    PRINT 'dbo.spError DELETED'
END
*/
DROP PROCEDURE IF EXISTS 
        dbo.spLog, 
        dbo.spInfo, 
        dbo.spWarn, 
        dbo.spError,
        dbo.spCreate_table_log,
        dbo.spFlush;
PRINT 'Cleanup complete.';

Cleanup complete.

In [4]:
CREATE OR ALTER PROCEDURE dbo.spCreate_table_log
    @DropIfExists BIT = 0
AS
BEGIN
    IF @DropIfExists = 1
    BEGIN
        DROP TABLE IF EXISTS dbo.tblLog;
        EXEC dbo.spInfo 'DELETED dbo.tblLog'
    END
    
    IF OBJECT_ID('dbo.tblLog', 'U') IS NULL
    BEGIN
        CREATE TABLE dbo.tblLog (
            -- TimeStamp DATETIME DEFAULT CONVERT(NVARCHAR, GETDATE(), 120),
            TimeStamp DATETIME DEFAULT GETDATE(),
            Level NVARCHAR(10),
            Message NVARCHAR(MAX),
            isDisplayed BIT DEFAULT 0
        )
        EXEC dbo.spInfo 'CREATED dbo.tblLog'
    END
END

The module 'spCreate_table_log' depends on the missing object 'dbo.spInfo'. The module will still be created; however, it cannot run successfully until the object exists.
The module 'spCreate_table_log' depends on the missing object 'dbo.spInfo'. The module will still be created; however, it cannot run successfully until the object exists.

## create tblLog

## spLog - base sp for logging

In [5]:
-- EXEC dbo.spCreate_table_log;
-- SELECT * FROM dbo.tblLog;

Commands completed successfully.

In [6]:
CREATE OR ALTER PROCEDURE dbo.spFLush
AS
BEGIN
    EXEC dbo.spCreate_table_log;    
    SELECT
        [Timestamp],
        [Level],
        [Message],
        [isDisplayed]
    FROM tblLog
    WHERE isDisplayed = 0;

    UPDATE dbo.tblLog
    SET [isDisplayed] = 1
    WHERE [isDisplayed] = 0;
END

Commands completed successfully.

In [7]:
CREATE OR ALTER PROCEDURE dbo.spLog
    @Level NVARCHAR(10),    -- failing to specify size defaults to 1 !!!
    @Message NVARCHAR(MAX),
    @Flush BIT = 0
AS
BEGIN
    EXEC dbo.spCreate_table_log;    
    INSERT INTO dbo.tblLog
    (Level, Message)
    SELECT
        -- GETDATE() as [Timestamp],
        @Level as [Level], 
        @Message as [Message];

    IF @Flush = 1
        EXEC dbo.spFlush

    DECLARE @Date NVARCHAR(20) = CONVERT(NVARCHAR, GETDATE(), 120) -- Style 120 is the "Golden Standard"
    SET @Level = UPPER(@Level)
    -- PRINT '@Date: ' + @Date;
    -- PRINT '@Level: ' + @Level;
    DECLARE @Msg NVARCHAR(MAX) = FORMATMESSAGE('%s | %s | %s', 
        @Date,
        @Level, 
        @Message
    );
    PRINT @Msg;
END
GO


Commands completed successfully.

In [8]:
EXEC dbo.spLog 'INFO', 'hello';
EXEC dbo.spLog 'INFO', 'hi', 1;

(1 row affected)
2026-03-27 07:27:59 | INFO | hello
(1 row affected)
(2 rows affected)
(2 rows affected)
2026-03-27 07:27:59 | INFO | hi

Timestamp               | Level | Message | isDisplayed
------------------------+-------+---------+------------
2026-03-27 07:27:59.003 | INFO  | hello   | 0          
2026-03-27 07:27:59.010 | INFO  | hi      | 0          
(2 rows)

## add info, warning, error helpers

In [9]:
CREATE OR ALTER PROCEDURE dbo.spInfo
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'INFO', @Message;
END
GO


Commands completed successfully.

In [10]:
CREATE OR ALTER PROCEDURE dbo.spWarn
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'WARN', @Message;
END
GO


Commands completed successfully.

In [11]:
CREATE OR ALTER PROCEDURE dbo.spError
    @Message NVARCHAR(MAX)
AS
BEGIN
    DECLARE @ErrorMsg NVARCHAR(MAX) = ''
    -- SELECT ERROR_NUMBER(), ERROR_MESSAGE();
    IF ERROR_MESSAGE() IS NOT NULL
        SET @ErrorMsg = FORMATMESSAGE('%s: %i - %s', 
            @Message, 
            ERROR_NUMBER(), 
            ERROR_MESSAGE()
        );
    ELSE
        SET @ErrorMsg = @Message;
    

    EXEC dbo.spLog 'ERROR', @ErrorMsg;
END
GO


Commands completed successfully.

## test sps

In [14]:
SELECT DB_NAME() AS db_name;
GO

PRINT 'start'
EXEC dbo.spLog @Level='INFO', @Message='hello'
EXEC dbo.spInfo @Message='hello'
EXEC dbo.spWarn @Message='hello'
EXEC dbo.spError @Message='hello'
PRINT 'end'
GO


(1 row affected)

db_name
-------
sample 
(1 row)

start
(1 row affected)
2026-03-27 07:30:04 | INFO | hello
(1 row affected)
2026-03-27 07:30:04 | INFO | hello
(1 row affected)
2026-03-27 07:30:04 | WARN | hello
(1 row affected)
2026-03-27 07:30:04 | ERROR | hello
end